# Capstone 4: AI-Powered Sprint Assistant

**Book:** [LLMs for Business & Quality Analysts](../index.html)

## Lab Overview
Build an **AI-Powered Sprint Assistant** -- a multi-function tool that helps BA/QA teams throughout the sprint lifecycle: planning, daily standups, story refinement, test coordination, and retrospectives.

## Prerequisites
- Python 3.9+
- OpenAI API key
- Completed Chapters 1-15 and Capstones 1-3


In [ ]:
# Setup
!pip install -q openai python-dotenv

import os
import json
from datetime import datetime, timedelta
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o'

## 1. Sprint Context Setup

Define the sprint context that the assistant will use across all its functions.


In [ ]:
# Sprint context
sprint = {
    'name': 'Sprint 15',
    'start_date': '2024-11-18',
    'end_date': '2024-11-29',
    'velocity_target': 34,
    'team': [
        {'name': 'Alice', 'role': 'BA', 'capacity_days': 9},
        {'name': 'Bob', 'role': 'QA Lead', 'capacity_days': 10},
        {'name': 'Charlie', 'role': 'Dev', 'capacity_days': 8},
        {'name': 'Diana', 'role': 'Dev', 'capacity_days': 10},
        {'name': 'Eve', 'role': 'QA', 'capacity_days': 10}
    ],
    'backlog': [
        {'id': 'US-301', 'title': 'Implement Apple Pay checkout', 'points': 8, 'status': 'To Do', 'assignee': 'Charlie', 'priority': 'High', 'dependencies': ['US-298']},
        {'id': 'US-302', 'title': 'Add order tracking notifications', 'points': 5, 'status': 'To Do', 'assignee': 'Diana', 'priority': 'Medium', 'dependencies': []},
        {'id': 'US-303', 'title': 'Search results pagination', 'points': 3, 'status': 'In Progress', 'assignee': 'Charlie', 'priority': 'Medium', 'dependencies': []},
        {'id': 'US-304', 'title': 'Customer review moderation', 'points': 5, 'status': 'To Do', 'assignee': 'Diana', 'priority': 'Low', 'dependencies': []},
        {'id': 'US-305', 'title': 'Fix cart total with tax calculation', 'points': 3, 'status': 'In Progress', 'assignee': 'Charlie', 'priority': 'Critical', 'dependencies': []},
        {'id': 'US-306', 'title': 'Add wishlist sharing via link', 'points': 5, 'status': 'To Do', 'assignee': 'Diana', 'priority': 'Low', 'dependencies': []},
        {'id': 'US-307', 'title': 'SSO integration with Google', 'points': 8, 'status': 'Blocked', 'assignee': 'Charlie', 'priority': 'High', 'dependencies': [], 'blocker': 'Waiting for Google OAuth credentials'}
    ],
    'defects_open': [
        {'id': 'BUG-401', 'severity': 'Critical', 'title': 'Payment fails for amounts > $9999', 'assigned_to': 'Charlie'},
        {'id': 'BUG-402', 'severity': 'High', 'title': 'Email notifications delayed by 30+ minutes', 'assigned_to': 'Diana'},
        {'id': 'BUG-403', 'severity': 'Medium', 'title': 'Search filter resets on page navigation', 'assigned_to': None}
    ]
}

total_points = sum(s['points'] for s in sprint['backlog'])
print(f"{sprint['name']}: {sprint['start_date']} to {sprint['end_date']}")
print(f"Backlog: {len(sprint['backlog'])} stories, {total_points} points (target: {sprint['velocity_target']})")
print(f"Open defects: {len(sprint['defects_open'])}")

## 2. Sprint Planning Assistant

Help the team plan the sprint with capacity analysis, risk assessment, and story sequencing.


In [ ]:
# Sprint planning analysis
planning_prompt = f"""As a Scrum Master AI, analyze this sprint and provide planning recommendations.

Sprint Data: {json.dumps(sprint)}

Provide:
1. Capacity Analysis: Can the team deliver the planned backlog? Show calculation.
2. Risk Assessment: Identify risks (blocked items, dependencies, overallocation).
3. Recommended Story Sequence: What order should stories be worked on?
4. Scope Recommendation: Should any stories be removed or added?
5. QA Coordination: When should testing start for each story?
6. Sprint Goal: Suggest a concise sprint goal.

Return a JSON object with:
- capacity_analysis: {{total_capacity_points, planned_points, utilization_pct, feasible}}
- risks: array of {{description, severity, mitigation}}
- recommended_sequence: ordered array of story IDs with reasoning
- scope_changes: array of {{action (add/remove/defer), story_id, reason}}
- qa_schedule: array of {{story_id, testing_start_day, estimated_qa_hours}}
- sprint_goal: string

Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': planning_prompt}], temperature=0.3
)
plan = json.loads(response.choices[0].message.content)

print(f"Sprint Goal: {plan['sprint_goal']}")
print(f"\nCapacity: {plan['capacity_analysis']['utilization_pct']}% utilized")
print(f"Feasible: {plan['capacity_analysis']['feasible']}")
print(f"\nRisks: {len(plan['risks'])}")
for r in plan['risks']:
    print(f"  [{r['severity']}] {r['description']}")
print(f"\nRecommended Sequence: {' -> '.join(plan['recommended_sequence'][:5])}...")

## 3. Daily Standup Analyzer

Process standup updates and generate insights and action items.


In [ ]:
# Simulated standup updates (Day 3 of sprint)
standup_updates = [
    {'person': 'Alice', 'yesterday': 'Refined acceptance criteria for US-301 and US-302', 'today': 'Starting requirements review for US-304', 'blockers': 'None'},
    {'person': 'Bob', 'yesterday': 'Wrote test cases for US-303', 'today': 'Testing US-305 (tax calculation fix)', 'blockers': 'Need test environment refresh'},
    {'person': 'Charlie', 'yesterday': 'Finished US-303 pagination, started US-305 bug fix', 'today': 'Continue US-305, then start US-301', 'blockers': 'US-307 still blocked on Google credentials'},
    {'person': 'Diana', 'yesterday': 'Working on US-302 notification service', 'today': 'Continue US-302, pick up BUG-402', 'blockers': 'None'},
    {'person': 'Eve', 'yesterday': 'Regression testing on last sprint items', 'today': 'Start testing US-303 (ready for QA)', 'blockers': 'BUG-403 not assigned yet'}
]

standup_prompt = f"""Analyze these daily standup updates and provide insights.

Sprint Context: {json.dumps(sprint)}
Day of Sprint: 3 of 10
Standup Updates: {json.dumps(standup_updates)}

Provide:
1. Progress Summary: Overall sprint health (on track / at risk / behind)
2. Burndown Status: Points completed vs expected at this point
3. Action Items: Specific follow-ups needed (with owners)
4. Dependency Alerts: Upcoming dependencies that need attention
5. Blockers Resolution: Priority order for resolving blockers
6. Team Coordination: Suggestions for pairing or handoffs

Format as a concise standup summary report."""

response = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': standup_prompt}], temperature=0.3
)
print('STANDUP ANALYSIS - Day 3')
print('=' * 60)
print(response.choices[0].message.content)

## 4. Sprint Retrospective Facilitator

Generate retrospective insights from sprint data and team feedback.


In [ ]:
# End-of-sprint data
sprint_results = {
    'planned_points': 37,
    'completed_points': 29,
    'stories_completed': ['US-302', 'US-303', 'US-305', 'US-304'],
    'stories_carried_over': ['US-301', 'US-306'],
    'stories_blocked': ['US-307'],
    'defects_found': 5,
    'defects_fixed': 8,
    'team_feedback': [
        {'person': 'Alice', 'went_well': 'Requirements for US-302 were very clear, dev had no questions', 'improve': 'Need more time for refinement before sprint starts'},
        {'person': 'Bob', 'went_well': 'Caught the tax bug early with boundary testing', 'improve': 'Test environment was down for 2 days, lost time'},
        {'person': 'Charlie', 'went_well': 'US-303 and US-305 delivered on time', 'improve': 'US-301 was underestimated, Apple Pay integration was more complex than expected'},
        {'person': 'Diana', 'went_well': 'Good collaboration with Bob on notification testing', 'improve': 'Too many context switches between stories and bug fixes'},
        {'person': 'Eve', 'went_well': 'Automated 5 new regression tests', 'improve': 'Late story handoffs from dev left little time for thorough testing'}
    ]
}

retro_prompt = f"""Facilitate a sprint retrospective analysis.

Sprint Plan: {json.dumps(sprint)}
Sprint Results: {json.dumps(sprint_results)}

Generate a structured retrospective report with:
1. Sprint Scorecard (velocity, completion rate, defect metrics)
2. What Went Well (themes from team feedback + data-driven observations)
3. What Needs Improvement (themes from feedback + data-driven observations)
4. Root Cause Analysis for carried-over stories
5. Action Items for next sprint (specific, assignable, measurable)
6. Process Improvement Experiments to try
7. Team Health indicators

Format as a professional retrospective document."""

response = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': retro_prompt}], temperature=0.4
)
print('SPRINT RETROSPECTIVE REPORT')
print('=' * 60)
print(response.choices[0].message.content)

In [ ]:
# Generate next sprint recommendations
next_sprint_prompt = f"""Based on the retrospective findings, generate recommendations for the next sprint.

Retrospective Data: {json.dumps(sprint_results)}
Team: {json.dumps(sprint['team'])}
Carried Over: {json.dumps([s for s in sprint['backlog'] if s['id'] in sprint_results['stories_carried_over']])}
Blocked: {json.dumps([s for s in sprint['backlog'] if s['id'] in sprint_results['stories_blocked']])}

Provide:
1. Recommended velocity target for next sprint
2. Capacity adjustments based on lessons learned
3. Stories to prioritize (carried over + blocked resolution)
4. Process changes to implement
5. QA strategy adjustments
6. Risks to watch

Return as JSON with: velocity_target, capacity_notes, priority_stories, process_changes, qa_adjustments, risks.
Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': next_sprint_prompt}], temperature=0.3
)
next_sprint = json.loads(response.choices[0].message.content)
print('NEXT SPRINT RECOMMENDATIONS')
print('=' * 60)
print(f"Velocity Target: {next_sprint['velocity_target']}")
print(f"Capacity Notes: {next_sprint['capacity_notes']}")
print(f"\nProcess Changes:")
for change in next_sprint['process_changes']:
    print(f"  - {change}")
print(f"\nQA Adjustments:")
for adj in next_sprint['qa_adjustments']:
    print(f"  - {adj}")

## Exercises
1. Build a Slack/Teams integration that posts the daily standup analysis automatically.
2. Add velocity trend tracking across multiple sprints to improve estimation accuracy.
3. Create a story refinement assistant that generates clarification questions during backlog grooming.
4. Build a sprint health dashboard using Streamlit that updates in real-time with AI insights.
